# LegalIR Task 1: Kaggle 2×T4 CUDA Smoke Gate (B1.1)
## UIT Data Science Challenge 2026 — Reproducible Training Workflow
**Pinned Git Commit:** `3792b13699f4706c5a698b2d147a55e97fe4c0ce`

### Smoke Test Purpose:
- **Fast & Deterministic (~3 minutes)** on free Kaggle T4 / 2×T4 GPU.
- Proves dataset integrity from `/kaggle/input/datasets/phucdangg/legalir-task1-clean-data`.
- Mines a small 50-query leakage-safe pair subset.
- Validates `BAAI/bge-reranker-v2-m3` + LoRA forward/backward pass on CUDA.
- Asserts finite loss ($L < \infty$) and parameter update ($\Delta w > 0$).
- Tests checkpoint save & reload into fresh model instance.
- Produces `kaggle_smoke_report.json` required before Colab A100 execution.

> **Note:** Zero PyTorch/CUDA reinstallation to avoid Kaggle environment breakage.


In [ ]:
# ==============================================================================
# Cell 1: Hardware Preflight & GPU Verification
# ==============================================================================
import os
import sys
import subprocess
import torch

print(f"[+] Python Version : {sys.version.split()[0]}")
print(f"[+] PyTorch Version: {torch.__version__}")
print(f"[+] CUDA Available : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("[!] WARNING: CUDA not detected. Ensure GPU accelerator is enabled in Kaggle settings.")
else:
    count = torch.cuda.device_count()
    print(f"[+] Visible CUDA Devices: {count}")
    for i in range(count):
        prop = torch.cuda.get_device_properties(i)
        vram = prop.total_memory / (1024**3)
        print(f"    - GPU {i}: {prop.name} | Total VRAM: {vram:.2f} GB")


In [ ]:
# ==============================================================================
# Cell 2: Repository Clone & Exact Git Commit Checkout
# ==============================================================================
from pathlib import Path

EXPECTED_COMMIT = os.environ.get("LEGALIR_COMMIT_SHA", "main")
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR = WORK_DIR / "LegalIR"

if not REPO_DIR.exists():
    # Check if already running from within the repository
    if (Path.cwd() / "src").is_dir() and (Path.cwd() / "scripts").is_dir():
        REPO_DIR = Path.cwd().resolve()
    else:
        print(f"[*] Cloning repository to {REPO_DIR}...")
        subprocess.run(["git", "clone", "https://github.com/silent9669/LegalIR.git", str(REPO_DIR)], check=True)

if (REPO_DIR / ".git").is_dir():
    try:
        subprocess.run(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR, check=False)
        if EXPECTED_COMMIT and EXPECTED_COMMIT != "main":
            subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT], cwd=REPO_DIR, check=True)
        else:
            subprocess.run(["git", "checkout", "main"], cwd=REPO_DIR, check=False)
            subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=False)
    except Exception as exc:
        print(f"[!] Warning checking out {EXPECTED_COMMIT} ({exc}). Using current branch.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
actual_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR).decode("utf-8").strip()
print(f"[+] Working in repository: {REPO_DIR} (commit: {actual_commit})")


In [ ]:
# ==============================================================================
# Cell 3: Minimal Dependencies Preflight (Zero PyTorch Reinstallation)
# ==============================================================================
# Remove incompatible preinstalled torchao if present
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
needed_packages = []
for mod, pkg in [
    ("bm25s", "bm25s"),
    ("pyvi", "pyvi"),
    ("peft", "peft"),
    ("accelerate", "accelerate"),
    ("sentencepiece", "sentencepiece"),
]:
    try:
        __import__(mod)
    except ImportError:
        needed_packages.append(pkg)

if needed_packages:
    print(f"[*] Installing missing packages: {needed_packages}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location"] + needed_packages, check=True)
print("[+] Dependency preflight completed.")


In [ ]:
# ==============================================================================
# Cell 4: Discover Dataset & Run Kaggle Smoke Test (scripts/run_kaggle_smoke.py)
# ==============================================================================
from src.data.canonical import discover_canonical_dataset_dir
from scripts.run_kaggle_smoke import run_kaggle_smoke

dataset_dir = discover_canonical_dataset_dir()
print(f"[+] Discovered Canonical Dataset: {dataset_dir}")

output_dir = WORK_DIR / "legalir_kaggle_smoke_out"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"[*] Running Smoke Pipeline on {dataset_dir}...")
report = run_kaggle_smoke(
    dataset_dir=dataset_dir,
    output_dir=output_dir,
    target_sha=EXPECTED_COMMIT,
    mock=False,
)
print(f"[+] Smoke execution completed with verdict: {report.get('verdict')}")


In [ ]:
# ==============================================================================
# Cell 5: Inspect Smoke Report & Assert PASS
# ==============================================================================
import json

report_path = output_dir / "kaggle_smoke_report.json"
if not report_path.is_file():
    raise FileNotFoundError(f"Smoke report missing at {report_path}")

report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps(report, indent=2, ensure_ascii=False))

assert report.get("verdict") == "PASS", f"Smoke gate failed: {report.get('error')}"
print("\n=================================================================")
print("[+] KAGGLE 2×T4 CUDA SMOKE GATE PASSED. READY FOR COLAB A100 RUN.")
print("=================================================================")
